<a href="https://colab.research.google.com/github/cbonnin88/EcoVolt-HR/blob/main/Compensation_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import gdown as gd

In [17]:
url_employees = 'https://drive.google.com/uc?id=10GJ4H7tVZpFu_hM6cegvpqUrR7bSXSg2'
url_compensation = 'https://drive.google.com/uc?id=1rZJgQSXrh68DX6cdm0NGRJhk_5xZumTT'

In [18]:
gd.download(url_employees,'employees.csv',quiet=True)

'employees.csv'

In [19]:
gd.download(url_compensation,'compensation.csv',quiet=True)

'compensation.csv'

In [20]:
ee = pl.read_csv('employees.csv')
comp = pl.read_csv('compensation.csv')

In [22]:
display(ee.head())

Employee_ID,first_name,last_name,Country,Department,Job_Title,Job_Level,Remote_Type,Hire_Date,Tenure_Years
str,str,str,str,str,str,str,str,str,f64
"""EV00001""","""Elif""","""van der Stael""","""The Netherlands""","""Grid Infrastructure""","""Substation Manager""","""T5""","""Remote""","""2022-05-10""",3.6
"""EV00002""","""Valentijn""","""Alpaidis""","""The Netherlands""","""Solar Engineering""","""Grid Connection Specialist""","""T3""","""Remote""","""2021-04-11""",4.7
"""EV00003""","""Henrik""","""Eriksen""","""Norway""","""Green Hydrogen R&D""","""Electrolysis Researcher""","""T5""","""Remote""","""2021-10-23""",4.2
"""EV00004""","""Joaquina""","""Anguita""","""Spain""","""Solar Engineering""","""PV Designer""","""T4""","""Remote""","""2020-07-10""",5.5
"""EV00005""","""Agnete""","""Andersen""","""Denmark""","""Grid Infrastructure""","""High Voltage Engineer""","""T4""","""Office""","""2022-05-06""",3.7


In [23]:
display(comp.head())

Employee_ID,Annual_Base_EUR,Market_Midpoint_EUR,Bonus_Target_Pct,Performance_Rating
str,f64,i64,f64,i64
"""EV00001""",51681.3,48000,0.05,3
"""EV00002""",107230.55,95000,0.2,3
"""EV00003""",46844.56,48000,0.2,3
"""EV00004""",80435.23,70000,0.1,2
"""EV00005""",67087.3,70000,0.1,2


JOIN: Inner Joihn on Employee_ID

In [24]:
df_comp = ee.join(comp, on='Employee_ID')

In [26]:
display(df_comp.schema)

Schema([('Employee_ID', String),
        ('first_name', String),
        ('last_name', String),
        ('Country', String),
        ('Department', String),
        ('Job_Title', String),
        ('Job_Level', String),
        ('Remote_Type', String),
        ('Hire_Date', String),
        ('Tenure_Years', Float64),
        ('Annual_Base_EUR', Float64),
        ('Market_Midpoint_EUR', Int64),
        ('Bonus_Target_Pct', Float64),
        ('Performance_Rating', Int64)])

In [27]:
display(df_comp.head())

Employee_ID,first_name,last_name,Country,Department,Job_Title,Job_Level,Remote_Type,Hire_Date,Tenure_Years,Annual_Base_EUR,Market_Midpoint_EUR,Bonus_Target_Pct,Performance_Rating
str,str,str,str,str,str,str,str,str,f64,f64,i64,f64,i64
"""EV00001""","""Elif""","""van der Stael""","""The Netherlands""","""Grid Infrastructure""","""Substation Manager""","""T5""","""Remote""","""2022-05-10""",3.6,51681.3,48000,0.05,3
"""EV00002""","""Valentijn""","""Alpaidis""","""The Netherlands""","""Solar Engineering""","""Grid Connection Specialist""","""T3""","""Remote""","""2021-04-11""",4.7,107230.55,95000,0.2,3
"""EV00003""","""Henrik""","""Eriksen""","""Norway""","""Green Hydrogen R&D""","""Electrolysis Researcher""","""T5""","""Remote""","""2021-10-23""",4.2,46844.56,48000,0.2,3
"""EV00004""","""Joaquina""","""Anguita""","""Spain""","""Solar Engineering""","""PV Designer""","""T4""","""Remote""","""2020-07-10""",5.5,80435.23,70000,0.1,2
"""EV00005""","""Agnete""","""Andersen""","""Denmark""","""Grid Infrastructure""","""High Voltage Engineer""","""T4""","""Office""","""2022-05-06""",3.7,67087.3,70000,0.1,2


In [31]:
display(df_comp.null_count())

Employee_ID,first_name,last_name,Country,Department,Job_Title,Job_Level,Remote_Type,Hire_Date,Tenure_Years,Annual_Base_EUR,Market_Midpoint_EUR,Bonus_Target_Pct,Performance_Rating
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [33]:
print(f'Number of Rows: {df_comp.shape[0]}')
print(f'Number of Columns: {df_comp.shape[1]}')

Number of Rows: 12000
Number of Columns: 14


# **Weighted Average Compa-Ratio**

- By Country and Department

In [34]:
# Creating a new column for Compa-Ratio
df_comp = df_comp.with_columns(
    (pl.col('Annual_Base_EUR') / pl.col('Market_Midpoint_EUR')).alias('Compa_Ratio')
)

In [35]:
display(df_comp.columns)

['Employee_ID',
 'first_name',
 'last_name',
 'Country',
 'Department',
 'Job_Title',
 'Job_Level',
 'Remote_Type',
 'Hire_Date',
 'Tenure_Years',
 'Annual_Base_EUR',
 'Market_Midpoint_EUR',
 'Bonus_Target_Pct',
 'Performance_Rating',
 'Compa_Ratio']

In [44]:
# Aggregation: Average Salary and Compa-Ratio by Department
dept_analysis = df_comp.group_by('Department').agg(
    pl.col('Annual_Base_EUR').mean().round(2).alias('Avg_Salary'),
    pl.col('Compa_Ratio').mean().round(2).alias('Avg_Compa'),
    pl.len().alias('Headcount')
).sort('Avg_Salary',descending=True)

Department,Avg_Salary,Avg_Compa,Headcount
str,f64,f64,u32
"""Leadership""",255475.03,1.06,6
"""Corporate""",75221.25,1.05,2474
"""Wind Operations""",74511.0,1.05,2380
"""Green Hydrogen R&D""",73714.42,1.05,2347
"""Grid Infrastructure""",73482.31,1.04,2357
"""Solar Engineering""",73428.5,1.05,2436


In [58]:
dept_comp = df_comp.group_by('Department').agg(
    (pl.col('Annual_Base_EUR').sum() / pl.col('Market_Midpoint_EUR').sum()).round(2).alias('Weighted_Avg_Compa')
).sort('Weighted_Avg_Compa',descending=True)

In [59]:
dept_comp = dept_comp.with_columns(
    pl.when(pl.col('Weighted_Avg_Compa') > 1.05).then(pl.lit('Over Market'))
    .when(pl.col('Weighted_Avg_Compa') < 0.95).then(pl.lit('Under Market'))
    .otherwise(pl.lit('On Target')).alias('Market_Alignment')
)

In [60]:
display(dept_comp)

Department,Weighted_Avg_Compa,Market_Alignment
str,f64,str
"""Leadership""",1.06,"""Over Market"""
"""Solar Engineering""",1.05,"""On Target"""
"""Wind Operations""",1.05,"""On Target"""
"""Green Hydrogen R&D""",1.05,"""On Target"""
"""Corporate""",1.05,"""On Target"""
"""Grid Infrastructure""",1.04,"""On Target"""


In [61]:
fig_bar = px.bar(
    dept_comp.to_pandas(),
    x='Department',
    y='Weighted_Avg_Compa',
    color='Market_Alignment',
    color_discrete_map={
        'Over Market':'#EF553B',
        'On Target': '#00CC96',
        'Under Market': '#636EFA'
    },
    title= 'Weighted Average Compa-Ratio by Department (EcoVolt Renewables)',
    labels={'Weighted_Avg_Compa':'Weighted Compa-Ratio'},
    text_auto= '.2f'
)

fig_bar.add_hline(y=1.0,
                line_dash='dash',
                line_color='black',
                annotation_text='Market Midpoint (1.0)',
                annotation_position='bottom right')

fig_bar.update_yaxes(range=[0.8,1.2])
fig_bar.update_layout(xaxis_tickangle=-45, template='plotly_white')

fig_bar.show()

In [62]:
fig_payband = px.box(
    df_comp.to_pandas(),
    x='Job_Level',
    y='Annual_Base_EUR',
    color='Remote_Type',
    category_orders={'Job_Level':['T1','T2','T3','T4','T5']},
    title='Salary Distributioin by Job Level and Work Mode',
    points='outliers'
)

fig_payband.update_layout(template='plotly_white')
fig_payband.show()

In [64]:
fig_density = px.density_contour(
    df_comp.to_pandas(),
    x='Tenure_Years',
    y='Compa_Ratio',
    marginal_x='histogram',
    marginal_y='histogram',
    title='Density Analysis: Tenure vs. Market Position',
    labels={'Compa_Ratio':'Ratio','Tenure_Years':'Tenure (Years)'}
)

fig_density.show()

# **Identifying Inequities (Action List)**

In [67]:
p25_df = df_comp.group_by('Job_Level').agg(
    pl.col('Annual_Base_EUR').quantile(0.25).alias('P25_Salary')
)

display(p25_df)

Job_Level,P25_Salary
str,f64
"""T4""",67792.29
"""T5""",46410.05
"""T2""",140359.65
"""T3""",91653.55
"""T1""",240310.07


In [69]:
action_list = df_comp.join(p25_df,on='Job_Level').filter(
    (pl.col('Performance_Rating') >= 4) &
    (pl.col('Annual_Base_EUR') < pl.col('P25_Salary'))
)

display(f'Identified {len(action_list)} High Performers eligible for a salary market adjustment')

'Identified 911 High Performers eligible for a salary market adjustment'